# Indicadores operativos + Análisis de rubros AFO

**PP2 · ITSE** — Input: `viviendas_procesadas.csv`, `organizaciones.csv`, `avance_rubros.csv`

KPIs para la toma de decisiones del ministerio, más el desglose del Avance Físico de Obra por rubro de construcción.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
DRIVE = Path('/content/drive/MyDrive/PP2')
Path('img').mkdir(exist_ok=True)

df = pd.read_csv(DRIVE / 'viviendas_procesadas.csv')
col_avance = 'avance_obra' if 'avance_obra' in df.columns else 'avanceObra'
col_estado = 'estado'
col_cuit   = 'cuit_org'
total = len(df)
print(f'Registros: {total}')

In [ ]:
# KPI 1 — Tasa de finalización  |  KPI 2 — Obras en riesgo
terminadas = df[col_estado].isin(['Finalizada','Adjudicada']).sum()
activas    = df[col_estado].isin(['Iniciada','Avanzada']).sum()
tasa_fin   = terminadas / total * 100
n_alto, n_medio, n_bajo = [(df['nivel_riesgo']==v).sum() for v in ['alto','medio','bajo']]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bars = axes[0].bar(['Terminadas','Activas'], [terminadas, activas],
                   color=['#22c55e','#3b82f6'], alpha=0.85, width=0.5)
axes[0].bar_label(bars, fmt='%d', padding=4, fontsize=12, fontweight='bold')
axes[0].set_title(f'KPI 1 — Tasa de finalización: {tasa_fin:.1f}%')

bars2 = axes[1].bar(['Bajo','Medio','Alto'], [n_bajo,n_medio,n_alto],
                    color=['#22c55e','#f59e0b','#ef4444'], alpha=0.85, width=0.5)
axes[1].bar_label(bars2, fmt='%d', padding=4, fontsize=11)
axes[1].set_title('KPI 2 — Obras por nivel de riesgo')

plt.tight_layout(); plt.savefig('img/04_kpi1_2.png', bbox_inches='tight'); plt.show()
print(f'Finalización: {tasa_fin:.1f}% | Riesgo alto: {n_alto} ({n_alto/total*100:.1f}%)')

In [ ]:
# KPI 3 — Tiempo promedio de ejecución
# Solo obras terminadas (Finalizada/Adjudicada):
# incluir obras en curso sesga el promedio porque su duración real es desconocida —
# una obra activa de 200 días podría terminar en 250 o en 700; cualquier valor es especulativo.
PLAZO = 90   # plazo contractual de construcción (días)
term_df = df[df[col_estado].isin(['Finalizada','Adjudicada'])].copy()
prom_d  = term_df['dias_activa'].mean()
med_d   = term_df['dias_activa'].median()
pct_sobre_plazo = (term_df['dias_activa'] > PLAZO).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(term_df['dias_activa'], bins=25, color='#6366f1', edgecolor='white', alpha=0.85)
axes[0].axvline(PLAZO,  color='#10b981', linestyle='--', linewidth=2, label=f'Plazo: {PLAZO}d')
axes[0].axvline(prom_d, color='#ef4444', linestyle='--', linewidth=2, label=f'Promedio: {prom_d:.0f}d')
axes[0].set_title('KPI 3 — Tiempo de ejecución (terminadas)')
axes[0].legend()

# Duración por clasificación: detecta qué tipos de vivienda requieren plazos distintos.
# Color: verde dentro del plazo (≤90), ámbar hasta 2× plazo (≤180), rojo más allá.
dur_clasif = term_df.groupby('clasificacion')['dias_activa'].mean().sort_values()
axes[1].barh(dur_clasif.index, dur_clasif.values,
             color=['#22c55e' if v<=PLAZO else '#f59e0b' if v<=2*PLAZO else '#ef4444' for v in dur_clasif.values],
             alpha=0.85)
axes[1].axvline(PLAZO, color='#64748b', linestyle='--', alpha=0.5)
axes[1].set_title('Duración promedio por clasificación')

plt.tight_layout(); plt.savefig('img/04_kpi3.png', bbox_inches='tight'); plt.show()
print(f'Promedio: {prom_d:.0f} días ({prom_d/30:.1f} meses) | Mediana: {med_d:.0f} días')
print(f'Superó el plazo de {PLAZO} días: {pct_sobre_plazo:.0f}% de las obras terminadas')

## Índice de Confiabilidad de ONG

Las ONGs son los ejecutores del programa bajo contrato. Este índice las ordena en una sola escala 0-100 para responder la pregunta de accountability: **¿a quién hay que auditar, a quién acompañar y a quién dejar trabajar?**

Combina tres dimensiones:
- **Avance promedio** (50%) — ¿entregan obra?
- **Ausencia de riesgo** (30%) — ¿sus obras se paralizan?
- **Honestidad del reporte** (20%) — ¿lo que declaran coincide con lo que verifica el técnico?

Lectura accionable: 🟢 ≥70 confiable · 🟡 50-70 seguimiento · 🔴 <50 auditar. Una ONG con índice bajo concentra obras estancadas y/o sobre-reporta su avance — es donde el ministerio debe intervenir primero.

In [ ]:
# Índice de Confiabilidad de ONG + cobertura geográfica
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}
try:
    df_orgs = pd.read_csv(DRIVE / 'organizaciones.csv')
    nombre_map = dict(zip(df_orgs['cuit'], df_orgs['nombre']))
except FileNotFoundError:
    nombre_map = {}
df_vis = pd.read_csv(DRIVE / 'visitas.csv')

# Índice 0-100 que combina tres dimensiones de la gestión de cada ONG:
#   avance promedio (entrega)           peso 0.50
#   ausencia de obras en riesgo         peso 0.30
#   honestidad del reporte              peso 0.20  (cuánto sobreestima vs. el técnico)
# La sobreestimación sale de cruzar cada visita con la ONG gestora de esa obra.
disc = (df_vis.merge(df[['num_exp', col_cuit]], left_on='vivienda_id', right_on='num_exp')
              .groupby(col_cuit)['diferencia_ong'].mean().clip(lower=0))

g = df[df[col_cuit].notna()].groupby(col_cuit)
conf = pd.DataFrame({
    'obras':      g.size(),
    'avance':     g[col_avance].mean(),
    'pct_riesgo': g['nivel_riesgo'].apply(lambda s: s.isin(['alto','medio']).mean()*100),
})
conf['sobreestim'] = disc.reindex(conf.index).fillna(0)
conf['confiab'] = (0.50*conf['avance']
                   + 0.30*(100 - conf['pct_riesgo'])
                   + 0.20*(100 - conf['sobreestim'].clip(0,15)/15*100)).round(1)
conf['nombre'] = [str(nombre_map.get(c, c))[:24] for c in conf.index]
conf = conf.sort_values('confiab')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Ranking de confiabilidad — verde ≥70 (confiable) | ámbar 50-70 (seguimiento) | rojo <50 (auditar)
tier = lambda v: C['ok'] if v>=70 else C['medio'] if v>=50 else C['alerta']
axes[0].barh(conf['nombre'], conf['confiab'], color=[tier(v) for v in conf['confiab']], alpha=0.9)
axes[0].bar_label(axes[0].containers[0], fmt='%.0f', padding=3)
axes[0].set_xlim(0, 100)
axes[0].set_title('Índice de Confiabilidad por ONG\nVerde ≥70 | Ámbar 50-70 | Rojo <50')

# Cobertura geográfica: obras activas por departamento
activas_df  = df[df[col_estado].isin(['Iniciada','Avanzada'])]
dep_activos = activas_df['departamento'].nunique()
dep_total   = df['departamento'].nunique()
cob = activas_df.groupby('departamento').agg(
    obras=(col_avance,'count'), riesgo=('nivel_riesgo', lambda x: (x=='alto').sum())
).sort_values('obras', ascending=False)
axes[1].bar(range(len(cob)), cob['obras'],
            color=[C['alerta'] if r > 0 else C['base'] for r in cob['riesgo']], alpha=0.85)
axes[1].set_xticks(range(len(cob)))
axes[1].set_xticklabels(cob.index, rotation=45, ha='right', fontsize=7)
axes[1].set_title(f'Obras activas por departamento ({dep_activos}/{dep_total})')

plt.tight_layout(); plt.savefig('img/04_confiab_geo.png', bbox_inches='tight'); plt.show()
print(conf[['nombre','obras','avance','pct_riesgo','sobreestim','confiab']].round(1).to_string(index=False))

In [ ]:
# Catálogo fijo (secuencia constructiva del sistema legacy VISOC — afo.jpeg).
# Pesos suman 100; cada rubro contribuye peso × (avance_rubro/100) al AFO total.
# Restricción clave: el rubro N solo puede iniciarse cuando el N-1 alcanzó el 100%.
RUBROS = {
    1:'Terreno y limpieza', 2:'Excavación e impermeabilización', 3:'Mampostería hasta dintel',
    4:'Mampostería cerámico/Block', 5:'Encadenado', 6:'Revoque interior', 7:'Revoque exterior',
    8:'Cielorraso con aislante', 9:'Construcción de cielorrasos', 10:'Carpintería',
    11:'Instalación de agua', 12:'Instalación eléctrica', 13:'Instalación sanitaria',
    14:'Revestimiento exterior', 15:'Varios'
}
PESOS = {1:3,2:5,3:10,4:10,5:5,6:10,7:8,8:5,9:4,10:10,11:7,12:8,13:7,14:5,15:3}
ORDEN = list(range(1, 16))

dr = pd.read_csv(DRIVE / 'avance_rubros.csv')
dr = dr.merge(df[['num_exp','nivel_riesgo','estado']], left_on='vivienda_id', right_on='num_exp', how='left')

# Pivotear a matriz (vivienda × rubro_id) para analizar la secuencia por fila
dr_p = dr.pivot(index='vivienda_id', columns='rubro_id', values='avance_pct').fillna(0)

# Verificar restricción secuencial: rubro[i+1] debe ser 0 si rubro[i] < 95%
# (95% como umbral para tolerar pequeñas imprecisiones de registro)
violaciones = sum(((dr_p[i] < 95) & (dr_p[i+1] > 5)).sum() for i in range(1, 15))
print(f'Verificación de secuencia: {violaciones} violaciones', end=' — ')
print('OK' if violaciones == 0 else 'REVISAR: hay obras con rubros fuera de orden')

# Etapa activa = primer rubro que no llegó al 98%
# Es el punto exacto de la cadena constructiva donde está detenida la obra
dr_p['etapa_activa']    = dr_p[ORDEN].apply(lambda row: next((r for r in ORDEN if row[r] < 98), 15), axis=1)
# Avance dentro de esa etapa: ¿llegó apenas al 5% o ya está en el 80%?
dr_p['avance_en_etapa'] = dr_p.apply(lambda r: r[int(r['etapa_activa'])], axis=1)

print(f'\nEtapas activas — distribución:')
print(dr_p['etapa_activa'].value_counts().sort_index().rename(RUBROS).to_string())

In [ ]:
# Merge con estado/riesgo para filtrar solo obras activas
dr_full = dr_p.merge(df[['num_exp','nivel_riesgo','estado']], left_index=True, right_on='num_exp', how='left')
activas = dr_full[dr_full['estado'].isin(['Iniciada','Avanzada'])].copy()

# Obras por etapa activa: ¿dónde está parada cada obra en la secuencia?
dist_etapa = activas['etapa_activa'].value_counts().reindex(ORDEN, fill_value=0)
dist_etapa.index = [RUBROS[i] for i in ORDEN]

# "Bloqueada" = obra en la etapa pero con avance < 30% dentro de ella.
# Distingue entre "avanzando despacio" y "paralizada al inicio del rubro".
activas['bloqueada'] = activas['avance_en_etapa'] < 30
bloqueo = (activas.groupby('etapa_activa')
           .agg(total=('avance_en_etapa','count'), bloqueadas=('bloqueada','sum'))
           .assign(pct_bloqueo=lambda d: (d['bloqueadas']/d['total']*100).round(1))
           .reindex(ORDEN).fillna(0))
bloqueo.index = [RUBROS[i] for i in ORDEN]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ¿Dónde se acumulan las obras?
umbral = dist_etapa.mean()
col_d = ['#ef4444' if v>=umbral*1.5 else '#f59e0b' if v>=umbral else '#22c55e' for v in dist_etapa]
axes[0].barh(dist_etapa.index, dist_etapa.values, color=col_d, alpha=0.85)
axes[0].bar_label(axes[0].containers[0], fmt='%.0f', padding=3, fontsize=9)
axes[0].set_title('Obras activas por etapa actual\n(rojo = cuello de botella por concentración)')
axes[0].set_xlabel('Cantidad de obras')

# ¿Cuáles están paralizada al inicio del rubro (sin avanzar)?
col_b = ['#ef4444' if v>=50 else '#f59e0b' if v>=25 else '#22c55e' for v in bloqueo['pct_bloqueo']]
axes[1].barh(bloqueo.index, bloqueo['pct_bloqueo'], color=col_b, alpha=0.85)
axes[1].bar_label(axes[1].containers[0], fmt='%.0f%%', padding=3, fontsize=9)
axes[1].set_title('% obras bloqueadas por etapa (avance < 30% dentro del rubro)\nRojo ≥50% | Amarillo 25–50% | Verde <25%')
axes[1].set_xlabel('% bloqueadas')

plt.tight_layout(); plt.savefig('img/04_etapa_activa.png', bbox_inches='tight'); plt.show()

etapa_cuello     = dist_etapa.idxmax()
etapa_mayor_bloqueo = bloqueo['pct_bloqueo'].idxmax()
print(f'Etapa con más obras en curso       : {etapa_cuello} ({dist_etapa.max():.0f} obras)')
print(f'Etapa con mayor tasa de bloqueo    : {etapa_mayor_bloqueo} ({bloqueo["pct_bloqueo"].max():.1f}% paralizadas)')

## El segundo cuello de botella: actas de finalización

El cuello de botella no es solo constructivo. Cuando una obra termina físicamente pasa a estado **Finalizada**, pero recién se considera cerrada cuando se tramita el **acta de finalización** y pasa a **Adjudicada** (entregada a la familia).

Ese trámite —responsabilidad compartida entre la ONG que lo inicia y el ministerio/técnico que lo genera— **se demora por falta de seguimiento**. El resultado: viviendas 100% construidas que quedan meses en limbo administrativo sin entregarse.

La diferencia con el cuello constructivo es **quién tiene la palanca**: la construcción es responsabilidad de la **organización gestora** (que materializa el beneficio, generalmente tercerizando los servicios), así que ante una obra trabada el ministerio solo puede reclamar y seguir de cerca a la gestora. El acta, en cambio, se destraba en gran parte **desde el propio ministerio** con gestión del trámite — por eso es el problema más rápido y barato de resolver de todo el programa.

In [ ]:
# Actas de finalización pendientes — cuello de botella administrativo
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}
UMBRAL = 180   # días esperando acta a partir de los cuales se considera "atascada"

hoy = pd.Timestamp.today()
df['fecha_fin_dt'] = pd.to_datetime(df['fecha_fin'], format='%d-%m-%Y', errors='coerce')

# Solo las Finalizada esperan acta. Las Adjudicada ya cerraron el ciclo administrativo.
fin = df[df[col_estado] == 'Finalizada'].copy()
fin['dias_espera'] = (hoy - fin['fecha_fin_dt']).dt.days
n_adj      = (df[col_estado] == 'Adjudicada').sum()
atascadas  = fin[fin['dias_espera'] > UMBRAL]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cuántas obras esperan acta y cuántas ya están atascadas
axes[0].bar(['Acta lista\n(Adjudicada)', 'En trámite\n(Finalizada)', 'Atascadas\n(>180 días)'],
            [n_adj, len(fin), len(atascadas)],
            color=[C['ok'], C['medio'], C['alerta']], alpha=0.9, width=0.6)
axes[0].bar_label(axes[0].containers[0], fmt='%d', padding=3, fontsize=11, fontweight='bold')
axes[0].set_title('Estado del trámite de actas')
axes[0].set_ylabel('Viviendas')

# Distribución del tiempo de espera del acta
axes[1].hist(fin['dias_espera'].dropna(), bins=25, color=C['base'], edgecolor='white', alpha=0.85)
axes[1].axvline(UMBRAL, color=C['alerta'], linestyle='--', linewidth=2, label=f'{UMBRAL}d (umbral atasco)')
axes[1].axvline(fin['dias_espera'].mean(), color=C['medio'], linestyle='--', linewidth=2,
                label=f'Media: {fin["dias_espera"].mean():.0f}d')
axes[1].set_title('Tiempo esperando el acta (obras Finalizadas)')
axes[1].set_xlabel('Días desde fin de obra'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('img/04_actas.png', bbox_inches='tight'); plt.show()
print(f'Viviendas construidas esperando acta : {len(fin)}')
print(f'Atascadas (>{UMBRAL} días sin seguimiento): {len(atascadas)} ({len(atascadas)/len(fin)*100:.0f}%)')
print(f'Espera promedio                      : {fin["dias_espera"].mean():.0f} días')

In [ ]:
# Cronograma de referencia: ¿a qué día debería estar terminada cada etapa?
# No hay fecha por rubro, así que se deriva del peso de cada rubro × el tiempo promedio
# real de una obra terminada (suma acumulada). Es una vara para detectar atrasos:
# una obra a día 300 que sigue en mampostería está retrasada frente a esta referencia.
C = {'base':'#4f46e5','ok':'#10b981','medio':'#f59e0b','alerta':'#f43f5e','neutro':'#94a3b8'}
PESOS  = {1:3,2:5,3:10,4:10,5:5,6:10,7:8,8:5,9:4,10:10,11:7,12:8,13:7,14:5,15:3}
RUBROS = {1:'Terreno',2:'Excavación',3:'Mampostería dintel',4:'Mampostería Block',5:'Encadenado',
          6:'Revoque interior',7:'Revoque exterior',8:'Cielorraso',9:'Cielorrasos',10:'Carpintería',
          11:'Inst. agua',12:'Inst. eléctrica',13:'Inst. sanitaria',14:'Revestimiento',15:'Varios'}

t_prom  = df[df[col_estado].isin(['Finalizada','Adjudicada'])]['dias_activa'].mean()
acc     = np.cumsum([PESOS[r] for r in range(1,16)]) / 100 * t_prom
nombres = [RUBROS[r] for r in range(1,16)]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.barh(nombres, acc, color=C['base'], alpha=0.85)
ax.bar_label(ax.containers[0], fmt='día %.0f', padding=3, fontsize=8)
ax.invert_yaxis()
ax.set_title(f'Cronograma de referencia — día esperado de fin de cada etapa (plazo medio: {t_prom:.0f}d)')
ax.set_xlabel('Días desde el inicio de la obra')
plt.tight_layout(); plt.savefig('img/04_cronograma.png', bbox_inches='tight'); plt.show()

In [ ]:
# Resumen ejecutivo + exportar
terminadas_df = df[df[col_estado].isin(['Finalizada','Adjudicada'])]
print('='*58 + '\nINDICADORES OPERATIVOS — PROGRAMA VIVSO\n' + '='*58)
print(f'  Total de viviendas             : {total}')
print(f'  Tasa de finalización           : {tasa_fin:.1f}%')
print(f'  Avance promedio (AFO)          : {df[col_avance].mean():.1f}%')
print(f'  Tiempo promedio ejecución      : {terminadas_df["dias_activa"].mean():.0f} días (plazo: 90)')
print(f'  Obras en riesgo alto           : {n_alto} ({n_alto/total*100:.1f}%)')
print(f'  Etapa cuello de botella        : {etapa_cuello} ({dist_etapa.max():.0f} obras)')
print(f'  Actas pendientes (construidas) : {len(fin)}')
print(f'  Actas atascadas (>180 días)    : {len(atascadas)} ({len(atascadas)/len(fin)*100:.0f}%)')
print(f'  ONG menos confiable            : {conf.iloc[0]["nombre"]} (índice {conf.iloc[0]["confiab"]:.0f})')
print('='*58)

# Exportar a Drive
ind_dep = (df.groupby('departamento')
             .agg(total=(col_avance,'count'), avance_prom=(col_avance,'mean'),
                  terminadas=(col_estado, lambda x: x.isin(['Finalizada','Adjudicada']).sum()),
                  riesgo_alto=('nivel_riesgo', lambda x: (x=='alto').sum()))
             .round(1).reset_index())
ind_dep.to_csv(DRIVE / 'indicadores_departamento.csv', index=False)
conf.reset_index().to_csv(DRIVE / 'indicadores_ong.csv', index=False)
print('Exportados: indicadores_departamento.csv + indicadores_ong.csv')